In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/Users/littlestars/Desktop/grain_project/data")

input_path = base / "final" / "final_panel_regression_ready_with_zone.csv"
output_path = base / "final" / "final_panel_regression_ready_with_zone_yield.csv"

df = pd.read_csv(input_path)

print("Columns:")
print(df.columns.tolist())

# 自动识别可能的产量列和面积列
possible_production_cols = ["production", "grain_production", "total_production", "prod", "Y"]
possible_area_cols = ["area_total", "sown_area", "grain_area", "area", "A"]

production_col = None
area_col = None

for c in possible_production_cols:
    if c in df.columns:
        production_col = c
        break

for c in possible_area_cols:
    if c in df.columns:
        area_col = c
        break

if production_col is None:
    raise ValueError("没有找到产量列。请检查列名，应该类似 production / grain_production。")

if area_col is None:
    raise ValueError("没有找到面积列。请检查列名，应该类似 area_total / sown_area / grain_area。")

print("Production column:", production_col)
print("Area column:", area_col)

# 清洗无效值
df = df.copy()
df[production_col] = pd.to_numeric(df[production_col], errors="coerce")
df[area_col] = pd.to_numeric(df[area_col], errors="coerce")

df = df[df[production_col].notna()]
df = df[df[area_col].notna()]
df = df[df[production_col] > 0]
df = df[df[area_col] > 0]

# 构造单产
df["yield"] = df[production_col] / df[area_col]

# 构造对数变量
df["ln_production"] = np.log(df[production_col])
df["ln_area"] = np.log(df[area_col])
df["ln_yield"] = np.log(df["yield"])

# 检查极端值
print("\nYield summary:")
print(df["yield"].describe())

# 保存
df.to_csv(output_path, index=False)

print("\nSaved to:")
print(output_path)

print("\nPreview:")
cols_to_show = []
for c in ["region", "year", production_col, area_col, "yield", "ln_yield"]:
    if c in df.columns:
        cols_to_show.append(c)

print(df[cols_to_show].head())

Columns:
['region', 'region_std', 'year', 'production', 'area_total', 'temp_annual_mean', 'prec_annual_sum', 'temp_grow_mean', 'prec_grow_sum', 'prec_winter_sum', 'temp_annual_lag1', 'prec_annual_lag1', 'temp_grow_lag1', 'prec_grow_lag1', 'ln_production', 'ln_area', 'temp_grow_sq', 'prec_grow_sq', 'agro_zone']
Production column: production
Area column: area_total

Yield summary:
count    1170.000000
mean     2273.048799
std      1183.000807
min       249.714112
25%      1461.499794
50%      1957.864640
75%      2868.823733
max      6666.666667
Name: yield, dtype: float64

Saved to:
/Users/littlestars/Desktop/grain_project/data/final/final_panel_regression_ready_with_zone_yield.csv

Preview:
           region  year  production  area_total        yield  ln_yield
0  Алтайский край  2007   4698310.0    3576.553  1313.641934  7.180559
1  Алтайский край  2008   3857490.0    3781.666  1020.050422  6.927607
2  Алтайский край  2009   5627845.0    3803.968  1479.466967  7.299437
3  Алтайский кра

In [20]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# 路径设置
# =========================

BASE_DIR = Path("/Users/littlestars/Desktop/grain_project")
DATA_DIR = BASE_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
INTER_DIR = DATA_DIR / "intermediate"
FINAL_DIR = DATA_DIR / "final"

SUBSIDY_DIR = Path("/Users/littlestars/Desktop/粮食补贴数据")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
INTER_DIR.mkdir(parents=True, exist_ok=True)

print("Project data path:", DATA_DIR)
print("Subsidy data path:", SUBSIDY_DIR)

Project data path: /Users/littlestars/Desktop/grain_project/data
Subsidy data path: /Users/littlestars/Desktop/粮食补贴数据


In [21]:
def clean_region_name(x):
    """基础清洗地区名称。"""
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("ё", "е").replace("Ё", "Е")
    return x


def find_col(df, candidates):
    """从候选列名中自动寻找真实列名。"""
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def safe_log(x):
    """只对正数取 log。"""
    x = pd.to_numeric(x, errors="coerce")
    return np.where(x > 0, np.log(x), np.nan)


def read_csv_safely(path):
    """自动尝试不同编码读取 CSV。"""
    for enc in ["utf-8", "utf-8-sig", "cp1251", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    raise ValueError(f"无法读取 CSV 文件: {path}")


def normalize_panel_keys(df):
    """统一地区列和年份列为 region, year。"""
    df = df.copy()

    region_col = find_col(df, [
        "region", "Region", "region_name", "name", "subject",
        "region_ru", "地区", "Регион", "Субъект"
    ])

    year_col = find_col(df, [
        "year", "Year", "年份", "год", "Год"
    ])

    if region_col is None:
        raise ValueError(f"找不到地区列。当前列名为: {df.columns.tolist()}")

    if year_col is None:
        raise ValueError(f"找不到年份列。当前列名为: {df.columns.tolist()}")

    df = df.rename(columns={region_col: "region", year_col: "year"})
    df["region"] = df["region"].apply(clean_region_name)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

    return df

In [22]:
candidate_files = [
    FINAL_DIR / "final_panel_regression_ready_with_zone.csv",
    FINAL_DIR / "final_panel_regression_ready.csv",
    FINAL_DIR / "final_panel_with_spi.csv",
]

main_path = None

for p in candidate_files:
    if p.exists():
        main_path = p
        break

if main_path is None:
    raise FileNotFoundError("在 data/final/ 中没有找到可用的最终面板数据。")

print("读取主面板文件:", main_path)

df = read_csv_safely(main_path)
df = normalize_panel_keys(df)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

读取主面板文件: /Users/littlestars/Desktop/grain_project/data/final/final_panel_regression_ready_with_zone.csv
Shape: (1170, 19)
Columns:
['region', 'region_std', 'year', 'production', 'area_total', 'temp_annual_mean', 'prec_annual_sum', 'temp_grow_mean', 'prec_grow_sum', 'prec_winter_sum', 'temp_annual_lag1', 'prec_annual_lag1', 'temp_grow_lag1', 'prec_grow_lag1', 'ln_production', 'ln_area', 'temp_grow_sq', 'prec_grow_sq', 'agro_zone']


,region,region_std,year,production,area_total,temp_annual_mean,prec_annual_sum,temp_grow_mean,prec_grow_sum,prec_winter_sum,temp_annual_lag1,prec_annual_lag1,temp_grow_lag1,prec_grow_lag1,ln_production,ln_area,temp_grow_sq,prec_grow_sq,agro_zone
0,Алтайский край,Алтайский край,2007,4698310.0,3576.553,4.392201,606.864057,14.966577,386.073528,99.304349,NaN,NaN,NaN,NaN,15.362713,8.182155,223.998427,149052.768805,black_soil
1,Алтайский край,Алтайский край,2008,3857490.0,3781.666,3.697218,518.462947,14.458340,317.014805,156.590101,4.392201,606.864057,14.966577,386.073528,15.165527,8.237920,209.043593,100498.386374,black_soil
2,Алтайский край,Алтайский край,2009,5627845.0,3803.968,2.031430,654.796853,13.675152,397.678262,196.547908,3.697218,518.462947,14.458340,317.014805,15.543237,8.243800,187.009776,158148.000344,black_soil
3,Алтайский край,Алтайский край,2010,4240800.0,3393.564,1.010635,517.690871,13.087710,252.172511,192.779893,2.031430,654.796853,13.675152,397.678262,15.260262,8.129636,171.288150,63590.975088,black_soil
4,Алтайский край,Алтайский край,2011,3919500.0,3628.322,2.650180,417.923236,14.869391,257.375904,194.746294,1.010635,517.690871,13.087710,252.172511,15.181475,8.196526,221.098799,66242.355883,black_soil


In [23]:
production_col = find_col(df, [
    "production", "grain_production", "total_production",
    "prod", "Y", "grain_output"
])

area_col = find_col(df, [
    "area_total", "sown_area", "grain_area", "area",
    "A", "sown_area_total"
])

print("Production column:", production_col)
print("Area column:", area_col)

if production_col is None:
    raise ValueError("没有找到产量列。请检查是否有 production / grain_production 等列。")

if area_col is None:
    raise ValueError("没有找到面积列。请检查是否有 area_total / sown_area / grain_area 等列。")

df[production_col] = pd.to_numeric(df[production_col], errors="coerce")
df[area_col] = pd.to_numeric(df[area_col], errors="coerce")

df = df[df[production_col] > 0].copy()
df = df[df[area_col] > 0].copy()

df["production_used"] = df[production_col]
df["area_used"] = df[area_col]

# 如果 production 是吨，area 是千公顷，则 production / area = kg/ha
df["yield_kg_ha"] = df["production_used"] / df["area_used"]
df["yield_t_ha"] = df["yield_kg_ha"] / 1000

df["ln_production"] = safe_log(df["production_used"])
df["ln_area"] = safe_log(df["area_used"])
df["ln_yield"] = safe_log(df["yield_kg_ha"])

df[["region", "year", "production_used", "area_used", "yield_kg_ha", "yield_t_ha", "ln_yield"]].head()

Production column: production
Area column: area_total


,region,year,production_used,area_used,yield_kg_ha,yield_t_ha,ln_yield
0,Алтайский край,2007,4698310.0,3576.553,1313.641934,1.313642,7.180559
1,Алтайский край,2008,3857490.0,3781.666,1020.050422,1.020050,6.927607
2,Алтайский край,2009,5627845.0,3803.968,1479.466967,1.479467,7.299437
3,Алтайский край,2010,4240800.0,3393.564,1249.659650,1.249660,7.130627
4,Алтайский край,2011,3919500.0,3628.322,1080.251422,1.080251,6.984949


In [24]:
df[["yield_kg_ha", "yield_t_ha"]].describe()

,yield_kg_ha,yield_t_ha
count,1170.000000,1170.000000
mean,2273.048799,2.273049
std,1183.000807,1.183001
min,249.714112,0.249714
25%,1461.499794,1.461500
50%,1957.864640,1.957865
75%,2868.823733,2.868824
max,6666.666667,6.666667


In [25]:
spi_cols_existing = [c for c in df.columns if "spi" in c.lower()]
print("主面板已有 SPI 变量:", spi_cols_existing)

if len(spi_cols_existing) == 0:
    spi_path = INTER_DIR / "climate_yearly_with_spi.csv"

    if spi_path.exists():
        print("读取 SPI 文件:", spi_path)

        spi_df = read_csv_safely(spi_path)

        # 你的 SPI 文件列名是 region_std, year, spi_grow_mean, spi_grow_min
        if "region_std" in spi_df.columns:
            spi_df = spi_df.rename(columns={"region_std": "region"})
        elif "region" not in spi_df.columns:
            raise ValueError(f"SPI 文件找不到地区列，当前列名为: {spi_df.columns.tolist()}")

        if "year" not in spi_df.columns:
            raise ValueError(f"SPI 文件找不到年份列，当前列名为: {spi_df.columns.tolist()}")

        spi_df["region"] = spi_df["region"].apply(clean_region_name)
        spi_df["year"] = pd.to_numeric(spi_df["year"], errors="coerce").astype("Int64")

        spi_cols = [c for c in spi_df.columns if "spi" in c.lower()]
        print("SPI 文件中的 SPI 变量:", spi_cols)

        keep_cols = ["region", "year"] + spi_cols
        spi_df = spi_df[keep_cols].drop_duplicates(["region", "year"])

        df = df.merge(spi_df, on=["region", "year"], how="left")

    else:
        print("没有找到 climate_yearly_with_spi.csv，将用降水构造 spi_grow_mean。")

        prec_col = find_col(df, [
            "prec_grow_sum", "grow_prec_sum", "precip_grow_sum",
            "prec_annual_sum", "precipitation"
        ])

        if prec_col is None:
            raise ValueError("没有 SPI 文件，也没有可用于构造 SPI 的降水列。")

        df[prec_col] = pd.to_numeric(df[prec_col], errors="coerce")

        df["spi_grow_mean"] = (
            df.groupby("region")[prec_col]
            .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
        )

        print("已构造 spi_grow_mean，来源变量:", prec_col)

spi_cols_now = [c for c in df.columns if "spi" in c.lower()]
print("当前 SPI 变量:", spi_cols_now)

df[["region", "year"] + spi_cols_now].head()

主面板已有 SPI 变量: []
读取 SPI 文件: /Users/littlestars/Desktop/grain_project/data/intermediate/climate_yearly_with_spi.csv
SPI 文件中的 SPI 变量: ['spi_grow_mean', 'spi_grow_min']
当前 SPI 变量: ['spi_grow_mean', 'spi_grow_min']


,region,year,spi_grow_mean,spi_grow_min
0,Алтайский край,2007,1.049620,-0.838191
1,Алтайский край,2008,0.114225,-0.451164
2,Алтайский край,2009,0.882891,-0.834674
3,Алтайский край,2010,-0.047701,-0.792166
4,Алтайский край,2011,-0.286207,-0.980573


In [26]:
temp_col = find_col(df, [
    "temp_grow_mean", "grow_temp_mean", "temperature_grow",
    "temp_annual_mean"
])

if temp_col is None:
    raise ValueError("没有找到温度变量，无法构造 SPEI-like。")

df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")

# 地区内部标准化温度
df["temp_grow_z"] = (
    df.groupby("region")[temp_col]
    .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
)

# 优先使用 spi_grow_mean
spi_main_col = find_col(df, [
    "spi_grow_mean", "SPI_grow_mean", "spi", "SPI", "spi_annual"
])

if spi_main_col is not None:
    df[spi_main_col] = pd.to_numeric(df[spi_main_col], errors="coerce")
    df["spei_grow_mean"] = df[spi_main_col] - df["temp_grow_z"]
    print(f"已构造 spei_grow_mean = {spi_main_col} - temp_grow_z")

else:
    prec_col = find_col(df, [
        "prec_grow_sum", "grow_prec_sum", "precip_grow_sum",
        "prec_annual_sum", "precipitation"
    ])

    if prec_col is None:
        raise ValueError("没有 SPI，也没有降水变量，无法构造 SPEI-like。")

    df[prec_col] = pd.to_numeric(df[prec_col], errors="coerce")

    df["prec_grow_z"] = (
        df.groupby("region")[prec_col]
        .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
    )

    df["spei_grow_mean"] = df["prec_grow_z"] - df["temp_grow_z"]
    print(f"已构造 spei_grow_mean = 标准化 {prec_col} - 标准化 {temp_col}")

df[["region", "year", temp_col, "temp_grow_z", "spei_grow_mean"]].head()

已构造 spei_grow_mean = spi_grow_mean - temp_grow_z


,region,year,temp_grow_mean,temp_grow_z,spei_grow_mean
0,Алтайский край,2007,14.966577,0.539200,0.510420
1,Алтайский край,2008,14.458340,-0.060094,0.174320
2,Алтайский край,2009,13.675152,-0.983601,1.866492
3,Алтайский край,2010,13.087710,-1.676290,1.628589
4,Алтайский край,2011,14.869391,0.424602,-0.710809


In [27]:
subsidy_files = list(SUBSIDY_DIR.glob("*.xlsx")) + list(SUBSIDY_DIR.glob("*.xls"))

print("补贴文件数量:", len(subsidy_files))

for p in subsidy_files:
    print(p.name)

补贴文件数量: 9
yjb05qcswwjgkbn0qhakja1cik1spxn5.xlsx
6d5635be18bc5bf56b159cee887ff95f.xlsx
c6bf6ceeac8c6178dd68592c8d31b188.xlsx
fpv8atuoyal61k43ejz91nxmum3r4egh.xlsx
d6q0yeso6kl5e1e7t75hzbup9hiomykb.xlsx
xhoulnrpq17fobwu5ljs81bn2v7ogsyv.xlsx
cf4b5c42b285af8837abc3d1f8628d17.xlsx
555d10328b12cb5737a79b1ea50f457c.xlsx
3248ff794dfaf3728d35081520fbd54c.xlsx


In [28]:
def extract_year_from_filename(path):
    """从文件名里提取年份。"""
    m = re.search(r"(20\d{2})", path.name)
    if m:
        return int(m.group(1))
    return None


def guess_year_from_excel(path):
    """如果文件名没有年份，就尝试从 Excel 前几行文字里找年份。"""
    try:
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            tmp = pd.read_excel(path, sheet_name=sheet, header=None, nrows=30)
            text = " ".join(tmp.astype(str).fillna("").values.ravel())
            m = re.search(r"(20\d{2})", text)
            if m:
                return int(m.group(1))
    except Exception:
        pass
    return None


def parse_one_subsidy_excel(path):
    """
    自动解析一个补贴 Excel。
    逻辑：
    1. 找年份；
    2. 找含俄文地区名称的列；
    3. 找数值列；
    4. 输出 region-year-subsidy_col_*。
    """
    year = extract_year_from_filename(path)
    if year is None:
        year = guess_year_from_excel(path)

    if year is None:
        print(f"跳过：无法识别年份 -> {path.name}")
        return None

    try:
        xls = pd.ExcelFile(path)
    except Exception as e:
        print(f"无法打开 {path.name}: {e}")
        return None

    frames = []

    for sheet in xls.sheet_names:
        try:
            raw = pd.read_excel(path, sheet_name=sheet, header=None)
        except Exception:
            continue

        raw = raw.dropna(how="all").dropna(axis=1, how="all")

        if raw.shape[0] < 5 or raw.shape[1] < 2:
            continue

        tmp = raw.copy()
        tmp.columns = [f"col_{i}" for i in range(tmp.shape[1])]

        # 找地区列：俄文字母最多的前几列
        candidate_region_cols = []

        for c in tmp.columns[:5]:
            s = tmp[c].astype(str).fillna("")
            score = s.str.contains(r"[А-Яа-я]", regex=True).sum()
            if score >= 5:
                candidate_region_cols.append((c, score))

        if not candidate_region_cols:
            continue

        candidate_region_cols = sorted(candidate_region_cols, key=lambda x: x[1], reverse=True)
        region_col = candidate_region_cols[0][0]

        # 找数值列
        numeric_cols = []

        for c in tmp.columns:
            if c == region_col:
                continue
            s = pd.to_numeric(tmp[c], errors="coerce")
            if s.notna().sum() >= 5:
                numeric_cols.append(c)

        if len(numeric_cols) == 0:
            continue

        out = pd.DataFrame()
        out["region"] = tmp[region_col].apply(clean_region_name)
        out["year"] = year

        for idx, c in enumerate(numeric_cols[:10]):
            out[f"subsidy_col_{idx+1}"] = pd.to_numeric(tmp[c], errors="coerce")

        value_cols = [c for c in out.columns if c.startswith("subsidy_col_")]

        out = out[out["region"].notna()]
        out = out[out["region"].str.len() >= 3]
        out = out[out[value_cols].notna().any(axis=1)]

        # 删除标题行、合计行、非地区行
        bad_patterns = [
            "итого",
            "всего",
            "наименование",
            "субъект",
            "российская федерация",
            "федеральный округ",
            "округ",
            "раздел",
            "показатель"
        ]

        mask_bad = pd.Series(False, index=out.index)

        for pat in bad_patterns:
            mask_bad = mask_bad | out["region"].str.lower().str.contains(pat, na=False)

        out = out[~mask_bad].copy()

        if len(out) > 0:
            frames.append(out)

    if len(frames) == 0:
        print(f"未解析到有效补贴表: {path.name}")
        return None

    res = pd.concat(frames, ignore_index=True)

    value_cols = [c for c in res.columns if c.startswith("subsidy_col_")]

    res = (
        res.groupby(["region", "year"], as_index=False)[value_cols]
        .sum(min_count=1)
    )

    print(f"已解析: {path.name}, year={year}, rows={len(res)}")

    return res

In [29]:
subsidy_frames = []

for p in subsidy_files:
    tmp = parse_one_subsidy_excel(p)
    if tmp is not None and len(tmp) > 0:
        subsidy_frames.append(tmp)

if len(subsidy_frames) == 0:
    raise ValueError("没有成功解析任何补贴文件。请检查补贴 Excel 的格式。")

subsidy_df = pd.concat(subsidy_frames, ignore_index=True)

subsidy_value_cols = [c for c in subsidy_df.columns if c.startswith("subsidy_col_")]

subsidy_df = (
    subsidy_df.groupby(["region", "year"], as_index=False)[subsidy_value_cols]
    .sum(min_count=1)
)

# 重命名几个主要补贴变量
rename_map = {}

if "subsidy_col_1" in subsidy_df.columns:
    rename_map["subsidy_col_1"] = "subsidy_main"

if "subsidy_col_2" in subsidy_df.columns:
    rename_map["subsidy_col_2"] = "subsidy_secondary"

if "subsidy_col_3" in subsidy_df.columns:
    rename_map["subsidy_col_3"] = "subsidy_third"

subsidy_df = subsidy_df.rename(columns=rename_map)

subsidy_df["region"] = subsidy_df["region"].apply(clean_region_name)
subsidy_df["year"] = pd.to_numeric(subsidy_df["year"], errors="coerce").astype("Int64")

print("补贴面板 shape:", subsidy_df.shape)
print("补贴年份:", subsidy_df["year"].min(), "-", subsidy_df["year"].max())
print("补贴地区数:", subsidy_df["region"].nunique())

subsidy_df.head()

已解析: yjb05qcswwjgkbn0qhakja1cik1spxn5.xlsx, year=2024, rows=93
已解析: 6d5635be18bc5bf56b159cee887ff95f.xlsx, year=2020, rows=95
已解析: c6bf6ceeac8c6178dd68592c8d31b188.xlsx, year=2019, rows=147
已解析: fpv8atuoyal61k43ejz91nxmum3r4egh.xlsx, year=2021, rows=89
已解析: d6q0yeso6kl5e1e7t75hzbup9hiomykb.xlsx, year=2023, rows=89
已解析: xhoulnrpq17fobwu5ljs81bn2v7ogsyv.xlsx, year=2022, rows=89
已解析: cf4b5c42b285af8837abc3d1f8628d17.xlsx, year=2016, rows=148
已解析: 555d10328b12cb5737a79b1ea50f457c.xlsx, year=2017, rows=147
已解析: 3248ff794dfaf3728d35081520fbd54c.xlsx, year=2018, rows=147
补贴面板 shape: (1044, 12)
补贴年份: 2016 - 2024
补贴地区数: 152


,region,year,subsidy_main,subsidy_secondary,subsidy_third,subsidy_col_4,subsidy_col_5,subsidy_col_6,subsidy_col_7,subsidy_col_8,subsidy_col_9,subsidy_col_10
0,Cубсидии на возмещение части процентной ставки...,2016,20.0,NaN,20.0,19.0,22.0,NaN,NaN,NaN,NaN,NaN
1,Cубсидии на возмещение части процентной ставки...,2017,20.0,NaN,20.0,19.0,22.0,NaN,NaN,NaN,NaN,NaN
2,Cубсидии на возмещение части процентной ставки...,2018,20.0,NaN,20.0,19.0,22.0,NaN,NaN,NaN,NaN,NaN
3,Cубсидии на возмещение части процентной ставки...,2019,20.0,NaN,20.0,19.0,22.0,NaN,NaN,NaN,NaN,NaN
4,Cубсидии на возмещение части процентной ставки...,2016,21.0,NaN,21.0,66.0,61.0,NaN,NaN,NaN,NaN,NaN


In [30]:
subsidy_output_path = INTER_DIR / "subsidy_panel_merged.csv"
subsidy_df.to_csv(subsidy_output_path, index=False, encoding="utf-8-sig")

print("补贴面板已保存到:", subsidy_output_path)

补贴面板已保存到: /Users/littlestars/Desktop/grain_project/data/intermediate/subsidy_panel_merged.csv


In [31]:
main_regions = set(df["region"].dropna().unique())
sub_regions = set(subsidy_df["region"].dropna().unique())

common_regions = main_regions & sub_regions
only_main = sorted(main_regions - sub_regions)
only_sub = sorted(sub_regions - main_regions)

print("主面板地区数:", len(main_regions))
print("补贴地区数:", len(sub_regions))
print("可直接匹配地区数:", len(common_regions))

print("\n主面板有、补贴没有的前 20 个:")
print(only_main[:20])

print("\n补贴有、主面板没有的前 20 个:")
print(only_sub[:20])

主面板地区数: 71
补贴地区数: 152
可直接匹配地区数: 70

主面板有、补贴没有的前 20 个:
['Республика Северная Осетия-Алания']

补贴有、主面板没有的前 20 个:
['Cубсидии на возмещение части процентной ставки по долгосрочным, среднесрочным и краткосрочным кредитам, взятым малыми формами хозяйствования', 'Cубсидии на возмещение части процентной ставки по инвестиционным кредитам (займам) на строительство и реконструкцию объектов для молочного скотоводства', 'Возмещение части затрат крестьянских (фермерских) хозяйств, включая индивидуальных предпринимателей, при оформлении в собственность используемых ими земельных участков из земель сельскохозяйственного назначения', 'Возмещение части затрат на закладку и уход за виноградниками', 'Возмещение части затрат на закладку и уход за многолетними плодовыми и ягодными насаждениями', 'Возмещение части затрат на раскорчевку выбывших из эксплуатации старых садов и рекультивацию раскорчеванных площадей', 'ДАЛЬНЕВОСТОЧНЫЙ Ф.О.', 'Донецкая Народная Республика', 'Еврейская автономная область', 'Запоро

In [32]:
merged = df.merge(
    subsidy_df,
    on=["region", "year"],
    how="left"
)

print("合并前 shape:", df.shape)
print("合并后 shape:", merged.shape)

subsidy_cols = [c for c in merged.columns if c.startswith("subsidy")]
print("补贴列:", subsidy_cols)

for c in subsidy_cols:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

if "subsidy_main" in merged.columns:
    merged["ln_subsidy_main"] = safe_log(merged["subsidy_main"])

    if "area_used" in merged.columns:
        merged["subsidy_per_area"] = merged["subsidy_main"] / merged["area_used"]
        merged["ln_subsidy_per_area"] = safe_log(merged["subsidy_per_area"])

print("subsidy_main 非缺失数量:", merged["subsidy_main"].notna().sum() if "subsidy_main" in merged.columns else "无")
print("ln_subsidy_main 非缺失数量:", merged["ln_subsidy_main"].notna().sum() if "ln_subsidy_main" in merged.columns else "无")

merged.head()

合并前 shape: (1170, 28)
合并后 shape: (1170, 38)
补贴列: ['subsidy_main', 'subsidy_secondary', 'subsidy_third', 'subsidy_col_4', 'subsidy_col_5', 'subsidy_col_6', 'subsidy_col_7', 'subsidy_col_8', 'subsidy_col_9', 'subsidy_col_10']
subsidy_main 非缺失数量: 529
ln_subsidy_main 非缺失数量: 529


,region,region_std,year,production,area_total,temp_annual_mean,prec_annual_sum,temp_grow_mean,prec_grow_sum,prec_winter_sum,...,subsidy_col_4,subsidy_col_5,subsidy_col_6,subsidy_col_7,subsidy_col_8,subsidy_col_9,subsidy_col_10,ln_subsidy_main,subsidy_per_area,ln_subsidy_per_area
0,Алтайский край,Алтайский край,2007,4698310.0,3576.553,4.392201,606.864057,14.966577,386.073528,99.304349,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Алтайский край,Алтайский край,2008,3857490.0,3781.666,3.697218,518.462947,14.458340,317.014805,156.590101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Алтайский край,Алтайский край,2009,5627845.0,3803.968,2.031430,654.796853,13.675152,397.678262,196.547908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Алтайский край,Алтайский край,2010,4240800.0,3393.564,1.010635,517.690871,13.087710,252.172511,192.779893,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Алтайский край,Алтайский край,2011,3919500.0,3628.322,2.650180,417.923236,14.869391,257.375904,194.746294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
merged = merged.sort_values(["region", "year"]).copy()

# 单产滞后项
merged["ln_yield_lag1"] = merged.groupby("region")["ln_yield"].shift(1)

# SPI 二次项
spi_col = find_col(merged, ["spi_grow_mean", "spi", "SPI", "spi_annual"])

if spi_col is not None:
    merged["spi_sq"] = merged[spi_col] ** 2
    print("使用 SPI 变量:", spi_col)
else:
    print("没有找到 SPI 变量。")

# SPEI-like 二次项
if "spei_grow_mean" in merged.columns:
    merged["spei_sq"] = merged["spei_grow_mean"] ** 2

# 补贴交互项
if "ln_subsidy_main" in merged.columns:
    if temp_col is not None:
        merged["temp_x_subsidy"] = merged[temp_col] * merged["ln_subsidy_main"]

    if spi_col is not None:
        merged["spi_x_subsidy"] = merged[spi_col] * merged["ln_subsidy_main"]

    if "spei_grow_mean" in merged.columns:
        merged["spei_x_subsidy"] = merged["spei_grow_mean"] * merged["ln_subsidy_main"]

merged.head()

使用 SPI 变量: spi_grow_mean


,region,region_std,year,production,area_total,temp_annual_mean,prec_annual_sum,temp_grow_mean,prec_grow_sum,prec_winter_sum,...,subsidy_col_10,ln_subsidy_main,subsidy_per_area,ln_subsidy_per_area,ln_yield_lag1,spi_sq,spei_sq,temp_x_subsidy,spi_x_subsidy,spei_x_subsidy
0,Алтайский край,Алтайский край,2007,4698310.0,3576.553,4.392201,606.864057,14.966577,386.073528,99.304349,...,NaN,NaN,NaN,NaN,NaN,1.101702,0.260529,NaN,NaN,NaN
1,Алтайский край,Алтайский край,2008,3857490.0,3781.666,3.697218,518.462947,14.458340,317.014805,156.590101,...,NaN,NaN,NaN,NaN,7.180559,0.013047,0.030387,NaN,NaN,NaN
2,Алтайский край,Алтайский край,2009,5627845.0,3803.968,2.031430,654.796853,13.675152,397.678262,196.547908,...,NaN,NaN,NaN,NaN,6.927607,0.779497,3.483793,NaN,NaN,NaN
3,Алтайский край,Алтайский край,2010,4240800.0,3393.564,1.010635,517.690871,13.087710,252.172511,192.779893,...,NaN,NaN,NaN,NaN,7.299437,0.002275,2.652304,NaN,NaN,NaN
4,Алтайский край,Алтайский край,2011,3919500.0,3628.322,2.650180,417.923236,14.869391,257.375904,194.746294,...,NaN,NaN,NaN,NaN,7.130627,0.081915,0.505250,NaN,NaN,NaN


In [34]:
key_vars = [
    "ln_yield",
    "ln_yield_lag1",
    "yield_t_ha",
    temp_col,
    spi_col,
    "spei_grow_mean",
    "subsidy_main",
    "ln_subsidy_main",
    "subsidy_per_area",
    "ln_subsidy_per_area"
]

key_vars = [c for c in key_vars if c is not None and c in merged.columns]

print("最终数据 shape:", merged.shape)
print("地区数:", merged["region"].nunique())
print("年份范围:", merged["year"].min(), "-", merged["year"].max())

print("\n核心变量描述统计:")
display(merged[key_vars].describe())

print("\n核心变量缺失比例:")
missing_ratio = merged[key_vars].isna().mean().sort_values(ascending=False)
display(missing_ratio)

最终数据 shape: (1170, 47)
地区数: 71
年份范围: 2007 - 2023

核心变量描述统计:


,ln_yield,ln_yield_lag1,yield_t_ha,temp_grow_mean,spi_grow_mean,spei_grow_mean,subsidy_main,ln_subsidy_main,subsidy_per_area,ln_subsidy_per_area
count,1170.000000,1099.000000,1170.000000,1170.000000,1170.000000,1170.000000,5.290000e+02,529.000000,5.290000e+02,529.000000
mean,7.596678,7.578418,2.273049,13.978768,0.254347,0.254347,1.074802e+11,22.761768,7.694834e+10,17.306297
std,0.526850,0.523920,1.183001,3.613353,0.677890,1.425612,9.212041e+10,6.291193,1.425283e+12,6.609665
min,5.520317,5.520317,0.249714,3.140021,-2.125848,-4.397661,6.000000e+01,4.094345,8.183122e-02,-2.503096
25%,7.287218,7.279361,1.461500,12.170721,-0.138294,-0.550994,2.600000e+10,23.981362,4.095918e+07,17.528087
50%,7.579610,7.565649,1.957865,14.237210,0.350961,0.361478,8.100000e+10,25.117715,1.948847e+08,19.087919
75%,7.961657,7.932238,2.868824,16.166216,0.695953,1.226944,1.830000e+11,25.932752,1.160766e+09,20.872346
max,8.804875,8.804875,6.666667,22.884283,2.028917,4.224985,2.940000e+11,26.406846,3.262500e+13,31.116100



核心变量缺失比例:


subsidy_main           0.547863
ln_subsidy_main        0.547863
subsidy_per_area       0.547863
ln_subsidy_per_area    0.547863
ln_yield_lag1          0.060684
ln_yield               0.000000
yield_t_ha             0.000000
temp_grow_mean         0.000000
spi_grow_mean          0.000000
spei_grow_mean         0.000000
dtype: float64

In [35]:
final_output_path = FINAL_DIR / "final_panel_yield_spi_spei_subsidy.csv"

merged.to_csv(final_output_path, index=False, encoding="utf-8-sig")

print("最终融合数据已保存到:")
print(final_output_path)

最终融合数据已保存到:
/Users/littlestars/Desktop/grain_project/data/final/final_panel_yield_spi_spei_subsidy.csv


In [36]:
reg_cols_main = ["ln_yield", temp_col, spi_col]
reg_cols_main = [c for c in reg_cols_main if c is not None]

reg_main = merged.dropna(subset=reg_cols_main)

print("主模型可用样本量:", reg_main.shape)
print("主模型地区数:", reg_main["region"].nunique())
print("主模型年份:", reg_main["year"].min(), "-", reg_main["year"].max())

if "ln_subsidy_main" in merged.columns:
    reg_subsidy = merged.dropna(subset=reg_cols_main + ["ln_subsidy_main"])
    print("\n加入补贴后的可用样本量:", reg_subsidy.shape)
    print("加入补贴后的地区数:", reg_subsidy["region"].nunique())
    print("加入补贴后的年份:", reg_subsidy["year"].min(), "-", reg_subsidy["year"].max())

主模型可用样本量: (1170, 47)
主模型地区数: 71
主模型年份: 2007 - 2023

加入补贴后的可用样本量: (529, 47)
加入补贴后的地区数: 69
加入补贴后的年份: 2016 - 2023
